# E20260905212645934620: E2 - DeBERTa full vs LoRA vs QLoRA

Этот template копируется командой `make new-experiment`. Вставьте код обучения в одну функцию и нажмите **Run All** — остальное обрабатывается автоматически.

In [ ]:
# Setup: найдите корень клонированного репозитория и импортируйте runner.
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "configs/project.json").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Откройте notebook внутри клонированного репозитория.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
if importlib.util.find_spec("clearml") is None:
    print("WARNING: установите ClearML один раз: pip install -e '.[tracking]'")
from pmldl_llm import (
    ExperimentOutput,
    load_experiment_setup,
    run_notebook_experiment,
)

PROJECT_ROOT

## 1. Готовый конфиг эксперимента

ID, owner, parent, seed, split и ClearML уже настроены командой `make new-experiment`. Эту ячейку менять не нужно.

In [ ]:
SETUP = load_experiment_setup("configs/experiments/E20260905212645934620.json", project_root=PROJECT_ROOT)
SETUP

## 2. Ваш эксперимент

Перенесите существующий код обучения и оценки внутрь функции. В конце верните итоговые validation-метрики и пути к файлам, которые нужно сохранить.

In [ ]:
def train_and_evaluate(run):
    import gc
    import json
    import math
    import os
    import platform
    import random
    import shutil
    import subprocess
    import sys
    import tempfile
    import time
    from contextlib import nullcontext
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        BitsAndBytesConfig,
        get_linear_schedule_with_warmup,
        set_seed,
    )
    from peft import (
        LoraConfig,
        TaskType,
        get_peft_model,
        prepare_model_for_kbit_training,
    )

    from pmldl_llm.config import validate_fold_roles
    from pmldl_llm.data import (
        decode_turns,
        load_checksum_manifest,
        load_competition_data,
        swap_probability_columns,
        swap_target_indices,
        target_indices,
        verify_competition_data_dir,
    )
    from pmldl_llm.evaluation import (
        evaluate_experiment_probabilities,
        normalize_probabilities,
    )
    from pmldl_llm.split import load_frozen_folds
    from pmldl_llm.truncation import balanced_head_tail_truncate

    seed = int(SETUP.seed)
    model_name = str(SETUP.model_name)
    model_revision = str(SETUP.model_revision)
    smoke = bool(SETUP.smoke_test)
    target_columns = ["winner_model_a", "winner_model_b", "winner_tie"]
    requested_max_length = 512
    fallback_max_length = 384
    smoke_max_length = 128
    effective_batch_size = 32
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
    h100_fast_path = "H100" in gpu_name.upper()
    micro_batch_size = 32 if h100_fast_path else (4 if torch.cuda.is_available() else 2)
    gradient_accumulation_steps = (
        effective_batch_size // micro_batch_size if not smoke else 2
    )
    eval_batch_size = 64 if h100_fast_path else (16 if torch.cuda.is_available() else 4)
    screening_steps = 500
    benchmark_steps = 20
    common_training = {
        "epochs": 1,
        "effective_batch_size": effective_batch_size,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "max_grad_norm": 1.0,
        "budget_weights": [1, 2, 2],
        "random_swap_probability": 0.5,
    }

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.set_float32_matmul_precision("high")

    data_candidates = []
    if os.environ.get("PMLDL_DATA_DIR"):
        data_candidates.append(Path(os.environ["PMLDL_DATA_DIR"]))
    data_candidates.extend(
        [
            PROJECT_ROOT / "data" / "llm-classification-finetuning",
            Path("/kaggle/input/llm-classification-finetuning"),
        ]
    )
    data_dir = next(
        (
            candidate
            for candidate in data_candidates
            if (candidate / "train.csv").is_file()
            and (candidate / "test.csv").is_file()
            and (candidate / "sample_submission.csv").is_file()
        ),
        None,
    )
    if data_dir is None:
        checked = ", ".join(str(path) for path in data_candidates)
        raise FileNotFoundError(
            "Competition data is unavailable. Attach/download train.csv, "
            "test.csv and sample_submission.csv, or set PMLDL_DATA_DIR. "
            f"Checked: {checked}"
        )

    checksum_manifest = PROJECT_ROOT / "data" / "checksums.sha256"
    expected_hashes = load_checksum_manifest(checksum_manifest)
    verified_hashes = verify_competition_data_dir(data_dir, expected_hashes)
    train_frame, _ = load_competition_data(data_dir)

    split_config_path = PROJECT_ROOT / "configs" / "split.json"
    split_config = json.loads(split_config_path.read_text(encoding="utf-8"))
    roles = validate_fold_roles(split_config)
    folds = load_frozen_folds(
        train_frame,
        PROJECT_ROOT / "data" / "splits" / "folds.csv",
        split_config_path,
        n_splits=roles.n_splits,
        metadata_path=PROJECT_ROOT / "data" / "splits" / "metadata.json",
        dataset_hashes=expected_hashes,
    )
    fold_values = folds["fold"].to_numpy()
    all_targets = target_indices(train_frame)
    train_mask = np.isin(fold_values, list(roles.training_folds))
    validation_mask = fold_values == roles.validation_fold
    if (train_mask & validation_mask).any():
        raise RuntimeError("Training and validation folds overlap.")
    if not train_mask.any() or not validation_mask.any():
        raise RuntimeError("Frozen training or validation selection is empty.")

    def deterministic_balanced_indices(mask, per_class):
        selected = []
        rng = np.random.default_rng(seed)
        for class_index in range(3):
            candidates = np.flatnonzero(mask & (all_targets == class_index))
            candidates = candidates.copy()
            rng.shuffle(candidates)
            selected.extend(candidates[:per_class].tolist())
        return np.asarray(sorted(selected), dtype=np.int64)

    if smoke:
        train_indices = deterministic_balanced_indices(train_mask, 24)
        validation_indices = deterministic_balanced_indices(validation_mask, 12)
    else:
        train_indices = np.flatnonzero(train_mask)
        validation_indices = np.flatnonzero(validation_mask)

    train_part = train_frame.iloc[train_indices].reset_index(drop=True)
    validation_part = train_frame.iloc[validation_indices].reset_index(drop=True)
    y_train = target_indices(train_part)
    y_validation = target_indices(validation_part)
    if set(train_part.columns).intersection({"fold"}):
        raise RuntimeError("Fold identifiers must not enter model features.")
    feature_columns = {"prompt", "response_a", "response_b"}
    if not feature_columns.issubset(train_part.columns):
        raise RuntimeError("Required text fields are absent.")
    if {"model_a", "model_b"}.intersection(feature_columns):
        raise RuntimeError("Model identity columns are forbidden as features.")

    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        bf16_supported = bool(torch.cuda.is_bf16_supported())
        compute_dtype = torch.bfloat16 if bf16_supported else torch.float16
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        device = torch.device("mps")
        bf16_supported = False
        compute_dtype = torch.float32
    else:
        device = torch.device("cpu")
        bf16_supported = False
        compute_dtype = torch.float32

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        revision=model_revision,
        use_fast=False,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token
    cls_token_id = tokenizer.cls_token_id
    if cls_token_id is None:
        cls_token_id = tokenizer.bos_token_id
    sep_token_id = tokenizer.sep_token_id
    if sep_token_id is None:
        sep_token_id = tokenizer.eos_token_id
    if cls_token_id is None or sep_token_id is None or tokenizer.pad_token_id is None:
        raise RuntimeError("Tokenizer must define CLS/BOS, SEP/EOS and PAD tokens.")

    def render_turns(value):
        rendered = []
        for index, turn in enumerate(decode_turns(value)):
            if turn is None:
                content = "null_response"
            elif isinstance(turn, str):
                content = turn
            else:
                content = json.dumps(
                    turn, ensure_ascii=False, sort_keys=True, separators=(",", ":")
                )
            rendered.append(f"<turn_{index}> {content}")
        return "\n<turn_boundary>\n".join(rendered)

    def encode_rows(frame, max_length, swap_flags=None):
        labels = target_indices(frame)
        if swap_flags is None:
            swap_flags = np.zeros(len(frame), dtype=bool)
        swap_flags = np.asarray(swap_flags, dtype=bool)
        if len(swap_flags) != len(frame):
            raise ValueError("swap_flags length does not match the frame.")
        sequences = []
        encoded_labels = labels.copy()
        for row_index, row in enumerate(frame.itertuples(index=False)):
            response_a = row.response_b if swap_flags[row_index] else row.response_a
            response_b = row.response_a if swap_flags[row_index] else row.response_b
            prompt_ids = tokenizer.encode(
                render_turns(row.prompt), add_special_tokens=False
            )
            response_a_ids = tokenizer.encode(
                render_turns(response_a), add_special_tokens=False
            )
            response_b_ids = tokenizer.encode(
                render_turns(response_b), add_special_tokens=False
            )
            prompt_ids, response_a_ids, response_b_ids, _ = (
                balanced_head_tail_truncate(
                    prompt_ids,
                    response_a_ids,
                    response_b_ids,
                    max_length=max_length,
                    special_tokens=4,
                    budget_weights=(1, 2, 2),
                )
            )
            input_ids = [
                cls_token_id,
                *prompt_ids,
                sep_token_id,
                *response_a_ids,
                sep_token_id,
                *response_b_ids,
                sep_token_id,
            ]
            if len(input_ids) > max_length:
                raise RuntimeError("Encoded sequence exceeds max_length.")
            sequences.append(input_ids)
        encoded_labels[swap_flags] = swap_target_indices(
            encoded_labels[swap_flags]
        )
        return sequences, encoded_labels

    class PreferenceDataset(Dataset):
        def __init__(self, sequences, labels=None):
            self.sequences = sequences
            self.labels = labels

        def __len__(self):
            return len(self.sequences)

        def __getitem__(self, index):
            item = {"input_ids": self.sequences[index]}
            if self.labels is not None:
                item["labels"] = int(self.labels[index])
            return item

    def collate(batch):
        longest = max(len(item["input_ids"]) for item in batch)
        if torch.cuda.is_available():
            longest = min(
                max_length_in_use,
                int(math.ceil(longest / 8.0) * 8),
            )
        input_ids = torch.full(
            (len(batch), longest),
            int(tokenizer.pad_token_id),
            dtype=torch.long,
        )
        attention_mask = torch.zeros((len(batch), longest), dtype=torch.long)
        labels = []
        for row_index, item in enumerate(batch):
            values = item["input_ids"]
            input_ids[row_index, : len(values)] = torch.tensor(
                values, dtype=torch.long
            )
            attention_mask[row_index, : len(values)] = 1
            if "labels" in item:
                labels.append(item["labels"])
        output = {"input_ids": input_ids, "attention_mask": attention_mask}
        if labels:
            output["labels"] = torch.tensor(labels, dtype=torch.long)
        return output

    def lora_settings(rank=16, dropout=0.05, learning_rate=1e-4):
        return {
            "r": int(rank),
            "alpha": 32,
            "dropout": float(dropout),
            "learning_rate": float(learning_rate),
        }

    screening_candidates = [
        ("reference", lora_settings()),
        ("rank_8", lora_settings(rank=8)),
        ("dropout_0", lora_settings(dropout=0.0)),
        ("learning_rate_2e_4", lora_settings(learning_rate=2e-4)),
    ]

    def reset_arm_seed():
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        set_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    def new_model(kind, settings):
        reset_arm_seed()
        quantization_active = kind == "qlora" and torch.cuda.is_available()
        load_kwargs = {
            "revision": model_revision,
            "num_labels": 3,
            "ignore_mismatched_sizes": True,
            "low_cpu_mem_usage": True,
        }
        # Master weights stay in fp32. Loading them in bf16 makes
        # AdamW updates of about 0.1% of a weight vanish into the
        # 8-bit mantissa, which stalls training; autocast below
        # still runs the forward pass in the compute dtype.
        if quantization_active:
            load_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=compute_dtype,
                # The pooler and the classifier are the modules LoRA keeps
                # trainable, and peft deep-copies them to do so. Copying a
                # Params4bit drops its quantisation state, so those two must
                # stay unquantised. Leaving the head out of 4 bits also
                # matches the full fine-tuning and LoRA arms.
                llm_int8_skip_modules=["pooler", "classifier"],
            )
            load_kwargs["device_map"] = {"": 0}
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            **load_kwargs,
        )
        model.config.pad_token_id = int(tokenizer.pad_token_id)
        model.config.use_cache = False
        if kind in {"lora", "qlora"}:
            if quantization_active:
                model = prepare_model_for_kbit_training(
                    model,
                    use_gradient_checkpointing=True,
                    gradient_checkpointing_kwargs={"use_reentrant": False},
                )
            lora_config = LoraConfig(
                task_type=TaskType.SEQ_CLS,
                r=int(settings["r"]),
                lora_alpha=int(settings["alpha"]),
                lora_dropout=float(settings["dropout"]),
                target_modules=["query_proj", "value_proj"],
                modules_to_save=["pooler", "classifier"],
                bias="none",
            )
            model = get_peft_model(model, lora_config)
            if quantization_active:
                # Linear4bit is not autocast aware: it returns its output in
                # the dtype of its input, so under autocast it hands back
                # fp32 while every other linear returns the compute dtype.
                # DeBERTa-v2 then takes its attention mask sentinel from an
                # fp32 query and applies it to bf16 scores, which overflows.
                # Casting the input makes the quantised layers behave like
                # the ones autocast already handles, and leaves the fp32
                # master weights of the other parameters untouched.
                def cast_quantised_input(module, args):
                    if args and torch.is_tensor(args[0]):
                        if args[0].dtype != compute_dtype:
                            return (args[0].to(compute_dtype),) + args[1:]
                    return None

                for submodule in model.modules():
                    if type(submodule).__name__ == "Linear4bit":
                        submodule.register_forward_pre_hook(
                            cast_quantised_input
                        )
        if not quantization_active:
            model.to(device)
            if hasattr(model, "gradient_checkpointing_enable"):
                try:
                    model.gradient_checkpointing_enable(
                        gradient_checkpointing_kwargs={"use_reentrant": False}
                    )
                except TypeError:
                    model.gradient_checkpointing_enable()
        return model, quantization_active

    def train_model(model, sequences, labels, learning_rate, namespace, max_steps=None):
        dataset = PreferenceDataset(sequences, labels)
        generator = torch.Generator()
        generator.manual_seed(seed)
        loader = DataLoader(
            dataset,
            batch_size=micro_batch_size,
            shuffle=True,
            generator=generator,
            num_workers=0,
            collate_fn=collate,
            pin_memory=device.type == "cuda",
        )
        if len(loader) == 0:
            raise RuntimeError("Training dataloader is empty.")
        if max_steps is None:
            optimizer_steps = math.ceil(
                len(loader) / gradient_accumulation_steps
            )
            microbatch_goal = len(loader)
        else:
            optimizer_steps = int(max_steps)
            microbatch_goal = optimizer_steps * gradient_accumulation_steps
        warmup_steps = int(math.ceil(optimizer_steps * common_training["warmup_ratio"]))
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
        optimizer = torch.optim.AdamW(
            parameters,
            lr=float(learning_rate),
            weight_decay=common_training["weight_decay"],
        )
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=optimizer_steps,
        )
        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=device.type == "cuda" and compute_dtype == torch.float16,
        )
        optimizer.zero_grad(set_to_none=True)
        model.train()
        curve = []
        optimizer_step = 0
        processed_microbatches = 0
        accumulated = 0
        loss_sum = 0.0
        started = time.perf_counter()
        while processed_microbatches < microbatch_goal:
            for batch in loader:
                if processed_microbatches >= microbatch_goal:
                    break
                batch = {
                    key: value.to(device, non_blocking=device.type == "cuda")
                    for key, value in batch.items()
                }
                autocast_context = (
                    torch.autocast(device_type="cuda", dtype=compute_dtype)
                    if device.type == "cuda"
                    else nullcontext()
                )
                with autocast_context:
                    output = model(**batch)
                    loss = output.loss
                scaler.scale(loss / gradient_accumulation_steps).backward()
                loss_sum += float(loss.detach().cpu())
                accumulated += 1
                processed_microbatches += 1
                is_last = processed_microbatches == microbatch_goal
                if accumulated == gradient_accumulation_steps or is_last:
                    scaler.unscale_(optimizer)
                    if accumulated != gradient_accumulation_steps:
                        correction = gradient_accumulation_steps / accumulated
                        for parameter in parameters:
                            if parameter.grad is not None:
                                parameter.grad.mul_(correction)
                    torch.nn.utils.clip_grad_norm_(
                        parameters,
                        common_training["max_grad_norm"],
                    )
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                    optimizer_step += 1
                    mean_loss = loss_sum / accumulated
                    curve.append(
                        {
                            "step": optimizer_step,
                            "loss": mean_loss,
                            "learning_rate": float(scheduler.get_last_lr()[0]),
                        }
                    )
                    if optimizer_step == 1 or optimizer_step % 10 == 0 or is_last:
                        run.log_metric(
                            "loss",
                            mean_loss,
                            namespace=namespace,
                            step=optimizer_step,
                        )
                        run.log_metric(
                            "learning_rate",
                            float(scheduler.get_last_lr()[0]),
                            namespace=namespace,
                            step=optimizer_step,
                        )
                    accumulated = 0
                    loss_sum = 0.0
            if max_steps is None:
                break
        elapsed = time.perf_counter() - started
        if optimizer_step != optimizer_steps:
            raise RuntimeError(
                f"Expected {optimizer_steps} optimizer steps, got {optimizer_step}."
            )
        return curve, elapsed

    def predict(model, sequences):
        loader = DataLoader(
            PreferenceDataset(sequences),
            batch_size=eval_batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=collate,
            pin_memory=device.type == "cuda",
        )
        probabilities = []
        model.eval()
        with torch.inference_mode():
            for batch in loader:
                batch = {
                    key: value.to(device, non_blocking=device.type == "cuda")
                    for key, value in batch.items()
                }
                autocast_context = (
                    torch.autocast(device_type="cuda", dtype=compute_dtype)
                    if device.type == "cuda"
                    else nullcontext()
                )
                with autocast_context:
                    logits = model(**batch).logits
                probabilities.append(F.softmax(logits.float(), dim=-1).cpu().numpy())
        if not probabilities:
            raise RuntimeError("Prediction dataloader is empty.")
        return normalize_probabilities(np.concatenate(probabilities, axis=0))

    def count_parameters(model):
        total = sum(parameter.numel() for parameter in model.parameters())
        trainable = sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )
        return int(trainable), int(total)

    def release_model(model):
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def write_json(relative_path, payload):
        path = run.artifact_path(relative_path)
        path.write_text(
            json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
            encoding="utf-8",
        )
        return path

    def save_checkpoint(model, arm_name):
        archive_base = run.artifact_path(f"models/{arm_name}_checkpoint")
        with tempfile.TemporaryDirectory(
            prefix=f"{arm_name}-",
            dir=run.artifact_dir,
        ) as temporary:
            checkpoint_dir = Path(temporary) / "checkpoint"
            model.save_pretrained(checkpoint_dir, safe_serialization=True)
            tokenizer.save_pretrained(checkpoint_dir)
            archive_path = shutil.make_archive(
                str(archive_base),
                "zip",
                root_dir=checkpoint_dir,
            )
        return Path(archive_path)

    swap_rng = np.random.default_rng(seed)
    training_swap_flags = swap_rng.random(len(train_part)) < 0.5

    max_length_in_use = smoke_max_length if smoke else requested_max_length
    train_sequences, encoded_y_train = encode_rows(
        train_part,
        max_length_in_use,
        training_swap_flags,
    )
    validation_sequences, _ = encode_rows(
        validation_part,
        max_length_in_use,
    )
    swapped_validation_sequences, _ = encode_rows(
        validation_part,
        max_length_in_use,
        np.ones(len(validation_part), dtype=bool),
    )
    benchmark = None

    if not smoke:
        benchmark_subset = min(
            len(train_sequences),
            micro_batch_size * gradient_accumulation_steps * benchmark_steps,
        )
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        benchmark_model, _ = new_model("lora", screening_candidates[0][1])
        _, benchmark_elapsed = train_model(
            benchmark_model,
            train_sequences[:benchmark_subset],
            encoded_y_train[:benchmark_subset],
            screening_candidates[0][1]["learning_rate"],
            "length_benchmark_train",
            max_steps=benchmark_steps,
        )
        projected_500_step_seconds = (
            benchmark_elapsed / benchmark_steps * screening_steps
        )
        benchmark = {
            "steps": benchmark_steps,
            "elapsed_seconds": float(benchmark_elapsed),
            "projected_500_step_seconds": float(projected_500_step_seconds),
            "tested_max_length": requested_max_length,
        }
        release_model(benchmark_model)
        if projected_500_step_seconds > 4 * 60 * 60:
            max_length_in_use = fallback_max_length
            train_sequences, encoded_y_train = encode_rows(
                train_part,
                max_length_in_use,
                training_swap_flags,
            )
            validation_sequences, _ = encode_rows(
                validation_part,
                max_length_in_use,
            )
            swapped_validation_sequences, _ = encode_rows(
                validation_part,
                max_length_in_use,
                np.ones(len(validation_part), dtype=bool),
            )
        benchmark["selected_max_length"] = max_length_in_use

    screening_results = []
    if smoke:
        selected_lora = screening_candidates[0][1]
        screening_results.append(
            {
                "name": "reference",
                "status": "selected_without_screening_in_smoke",
                **selected_lora,
            }
        )
    else:
        for candidate_name, candidate_settings in screening_candidates:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            candidate_model, _ = new_model("lora", candidate_settings)
            curve, train_seconds = train_model(
                candidate_model,
                train_sequences,
                encoded_y_train,
                candidate_settings["learning_rate"],
                f"screen_{candidate_name}_train",
                max_steps=screening_steps,
            )
            original_probability = predict(candidate_model, validation_sequences)
            swapped_probability = predict(
                candidate_model, swapped_validation_sequences
            )
            swapped_back_probability = swap_probability_columns(
                swapped_probability
            )
            candidate_metrics = evaluate_experiment_probabilities(
                y_validation,
                original_probability,
                swapped_back_probability,
            )
            peak_memory_mb = (
                float(torch.cuda.max_memory_allocated() / (1024 ** 2))
                if torch.cuda.is_available()
                else 0.0
            )
            result = {
                "name": candidate_name,
                **candidate_settings,
                "optimizer_steps": screening_steps,
                "train_runtime_seconds": float(train_seconds),
                "peak_gpu_memory_mb": peak_memory_mb,
                "metrics": candidate_metrics,
                "last_train_loss": float(curve[-1]["loss"]),
            }
            screening_results.append(result)
            run.log_metrics(
                {
                    **candidate_metrics,
                    "train_runtime_seconds": train_seconds,
                    "peak_gpu_memory_mb": peak_memory_mb,
                },
                namespace=f"screen_{candidate_name}_validation",
                step=0,
            )
            release_model(candidate_model)
        selected_screen = min(
            screening_results,
            key=lambda item: float(item["metrics"]["log_loss"]),
        )
        selected_lora = {
            key: selected_screen[key]
            for key in ("r", "alpha", "dropout", "learning_rate")
        }

    arm_specs = [
        (
            "full_ft",
            "full",
            {
                "learning_rate": 2e-5,
                "r": 0,
                "alpha": 0,
                "dropout": 0.0,
            },
        ),
        ("lora", "lora", dict(selected_lora)),
        ("qlora", "qlora", dict(selected_lora)),
    ]
    arm_results = {}
    artifacts = {}

    for arm_name, kind, settings in arm_specs:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        arm_started = time.perf_counter()
        model, quantization_active = new_model(kind, settings)
        trainable_parameters, total_parameters = count_parameters(model)
        arm_max_steps = 2 if smoke else None
        curve, train_seconds = train_model(
            model,
            train_sequences,
            encoded_y_train,
            settings["learning_rate"],
            f"{arm_name}_train",
            max_steps=arm_max_steps,
        )
        original_probability = predict(model, validation_sequences)
        swapped_probability = predict(model, swapped_validation_sequences)
        swapped_back_probability = swap_probability_columns(swapped_probability)
        metrics = evaluate_experiment_probabilities(
            y_validation,
            original_probability,
            swapped_back_probability,
        )
        arm_runtime_seconds = time.perf_counter() - arm_started
        peak_memory_mb = (
            float(torch.cuda.max_memory_allocated() / (1024 ** 2))
            if torch.cuda.is_available()
            else 0.0
        )
        averaged_probability = normalize_probabilities(
            0.5 * (original_probability + swapped_back_probability)
        )

        prediction_path = run.artifact_path(
            f"predictions/{arm_name}_validation.csv"
        )
        prediction_frame = pd.DataFrame(
            {
                "id": validation_part["id"].to_numpy(),
                "y_true": y_validation,
                "original_winner_model_a": original_probability[:, 0],
                "original_winner_model_b": original_probability[:, 1],
                "original_winner_tie": original_probability[:, 2],
                "swapped_back_winner_model_a": swapped_back_probability[:, 0],
                "swapped_back_winner_model_b": swapped_back_probability[:, 1],
                "swapped_back_winner_tie": swapped_back_probability[:, 2],
                "winner_model_a": averaged_probability[:, 0],
                "winner_model_b": averaged_probability[:, 1],
                "winner_tie": averaged_probability[:, 2],
            }
        )
        prediction_frame.to_csv(prediction_path, index=False)
        curve_path = write_json(f"curves/{arm_name}.json", curve)
        artifacts[f"predictions/{arm_name}_validation.csv"] = prediction_path
        artifacts[f"curves/{arm_name}.json"] = curve_path
        if not smoke:
            checkpoint_path = save_checkpoint(model, arm_name)
            artifacts[f"models/{arm_name}_checkpoint.zip"] = checkpoint_path

        arm_result = {
            "kind": kind,
            "settings": settings,
            "metrics": metrics,
            "trainable_parameters": trainable_parameters,
            "total_parameters": total_parameters,
            "trainable_fraction": float(
                trainable_parameters / max(total_parameters, 1)
            ),
            "optimizer_steps": len(curve),
            "train_runtime_seconds": float(train_seconds),
            "runtime_seconds": float(arm_runtime_seconds),
            "peak_gpu_memory_mb": peak_memory_mb,
            "quantization_requested": kind == "qlora",
            "quantization_active": quantization_active,
        }
        arm_results[arm_name] = {
            **arm_result,
            "original_probability": original_probability,
            "swapped_back_probability": swapped_back_probability,
        }
        run.log_metrics(
            {
                **metrics,
                "trainable_parameters": trainable_parameters,
                "total_parameters": total_parameters,
                "trainable_fraction": arm_result["trainable_fraction"],
                "train_runtime_seconds": train_seconds,
                "runtime_seconds": arm_runtime_seconds,
                "peak_gpu_memory_mb": peak_memory_mb,
                "quantization_active": float(quantization_active),
            },
            namespace=f"{arm_name}_validation",
            step=0,
        )
        release_model(model)

    best_arm = min(
        arm_results,
        key=lambda name: float(arm_results[name]["metrics"]["log_loss"]),
    )
    serializable_arms = {
        name: {
            key: value
            for key, value in result.items()
            if key not in {"original_probability", "swapped_back_probability"}
        }
        for name, result in arm_results.items()
    }
    study_payload = {
        "experiment_id": SETUP.experiment_id,
        "smoke_test": smoke,
        "seed": seed,
        "model": {"name": model_name, "revision": model_revision},
        "protocol": {
            **common_training,
            "split_seed": roles.seed,
            "training_folds": list(roles.training_folds),
            "validation_fold": roles.validation_fold,
            "calibration_fold_unopened": roles.calibration_fold,
            "final_holdout_fold_unopened": roles.final_holdout_fold,
            "train_rows": int(len(train_part)),
            "validation_rows": int(len(validation_part)),
            "max_length": max_length_in_use,
            "micro_batch_size": micro_batch_size,
            "gradient_accumulation_steps": gradient_accumulation_steps,
            "null_sentinel": "null_response",
            "turn_boundary": "<turn_boundary>",
            "model_identity_features_used": False,
        },
        "benchmark": benchmark,
        "screening": screening_results,
        "selected_lora": selected_lora,
        "arms": serializable_arms,
        "best_arm": best_arm,
    }
    study_path = write_json("study.json", study_payload)
    artifacts["study.json"] = study_path

    try:
        freeze_output = subprocess.run(
            [sys.executable, "-m", "pip", "freeze"],
            check=True,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        ).stdout.splitlines()
    except (OSError, subprocess.CalledProcessError) as error:
        freeze_output = [f"unavailable: {type(error).__name__}: {error}"]

    environment_payload = {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": {
            "torch": torch.__version__,
            "transformers": __import__("transformers").__version__,
            "peft": __import__("peft").__version__,
            "accelerate": __import__("accelerate").__version__,
            "bitsandbytes": __import__("bitsandbytes").__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
        "pip_freeze": freeze_output,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": (
            torch.backends.cudnn.version() if torch.cuda.is_available() else None
        ),
        "gpu_names": [
            torch.cuda.get_device_name(index)
            for index in range(torch.cuda.device_count())
        ],
        "mps_available": bool(
            getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available()
        ),
        "requested_compute_dtype": "bfloat16",
        "actual_compute_dtype": str(compute_dtype),
        "bf16_supported": bf16_supported,
        "model": {"name": model_name, "revision": model_revision},
        "verified_data_hashes": verified_hashes,
    }
    environment_path = write_json("environment.json", environment_payload)
    artifacts["environment.json"] = environment_path

    best = arm_results[best_arm]
    return ExperimentOutput.from_predictions(
        y_true=y_validation,
        original_probabilities=best["original_probability"],
        swapped_back_probabilities=best["swapped_back_probability"],
        artifacts=artifacts,
    )

## 3. Автоматический запуск

Эту ячейку менять не нужно. При ошибке run автоматически сохранится со статусом `failed`; при успехе он будет проверен и попадёт в ClearML, а full-run — в локальный leaderboard.

In [ ]:
RESULT = run_notebook_experiment(
    train_and_evaluate,
    SETUP,
    project_root=PROJECT_ROOT,
)
RESULT

## Что получится

- `configs/experiments/<experiment_id>.json` — созданный конфиг;
- `results/runs/<run_id>/` — проверенные метрики и metadata;
- `artifacts/<run_id>/` — модели и predictions;
- ClearML task — live-метрики;
- `results/leaderboard.csv` — автоматически обновлённое локальное сравнение.

Первый **Run All** — smoke-test. Затем выполните `make prepare-full EXPERIMENT=<ID>`, перезапустите kernel и снова нажмите **Run All**. После успеха выполните `make submit-experiment EXPERIMENT=<ID>`.